# 17 — Contratos de eventos e replay causal local

Como manter as features e decisões compatíveis com a informação disponível no
instante de cada transação, e recuperar uma execução interrompida?

A checagem isolada de serving não justificou introduzir Prefect; essa entrega
foi retirada. Agora implementamos uma capacidade operacional: transação sem
rótulo, feedback separado sete dias depois e estado local recuperável.


## Decisão e fronteiras

`streaming/contracts/` define eventos estritos; `streaming/features/` produz os
19 atributos do Gold; `streaming/replay/` coordena a execução local. SQLite
mantém transações, feedback, decisões, eventos aceitos e checkpoint juntos.

Empates excluem todos os peers do histórico. Feedback disponível exatamente em
t só entra depois de t. Duplicata idêntica retorna a decisão salva; conflito ou
evento novo anterior ao checkpoint é rejeitado, sem alteração retroativa.
O relógio sem timezone é preservado; não inferimos um fuso para a fonte.


## O que foi conferido

Os checks compararam os atributos incrementais com o Gold em DuckDB e scores
com o mesmo modelo sintético pontuando o lote offline. Também cobriram rótulos
ausentes, históricos vazios/zero, nanossegundos, troca de ordem entre peers,
duplicatas/conflitos, eventos tardios, mudança de identidade e dois writers.

Uma falha de scorer e uma saída abrupta de processo durante a pontuação não
publicaram evento, decisão ou checkpoint parcial. A retomada produziu uma
única decisão confirmada. A CLI deixou seis eventos, retomou para 20 eventos
únicos e dez decisões, e repetiu 22 entregas sem novas gravações.

[Recibo de preparação](../../references/evidence/causal_replay_preparation_2026-10-13.json).
Os checks não avaliaram a referência nativa nem um período novo do Handbook.


## Validação no ambiente do autor — 2026-10-13

O autor aprovou 448 testes/93 avisos e tox `py314`, commitando a implementação
em `23c863e`. A CLI processou seis eventos, retomou para 20 únicos/dez requests
de atributos/dez feedbacks e repetiu 22 entregas sem novas gravações.
`inspect` confirmou os contadores nas saídas fornecidas. Hashes de código e
entrada coincidem com a preparação; SQLite no autor foi 3.46.1.

O demo manteve `scoring: null`: não pontuou a referência real nem comparou
paridade nessa invocação. A suíte cobriu paridade com dados/release sintéticos.
Publicação, CI e revisão da branch ficam pendentes antes da integração.
16–30/09 continua reservado; o próximo protocolo fixará contexto, release e
política de investigação antes de seu acesso.

[Recibo do relato](../../references/evidence/causal_replay_author_validation_2026-10-13.json).


## Limites e próxima ação

A CLI demonstra fixtures e calcula atributos; scoring opcional foi conferido
na biblioteca com uma release gerada. Não há broker, features publicadas em
serviço compartilhado, fila de investigação instantânea ou monitoramento.
O banco retém histórico; desempenho e retenção ainda precisam de medição/política.
Repetição idempotente no banco não garante entrega exatamente uma vez fora dele.

16–30/09 permanece reservado. Antes de acessá-lo, fixar protocolo de janela,
contexto de aquecimento, release, emissão de scores e política de investigação.
Não converter silenciosamente o ranking retrospectivo diário em decisão online.

[Comandos, recuperação e regras](../../docs/operations/CAUSAL_REPLAY.md).
[Próximos critérios](../../docs/project/ROADMAP.md#próxima-entrega-concreta).


## Próxima decisão operacional

O replay local e o relato do autor foram integrados pelo PR #4. Streaming
é o objetivo central de entrega: scores por evento e triagem próxima do
tempo real. A política preparada alerta no primeiro cruzamento de um
limiar, até cem clientes distintos/dia, sem substituir alertas anteriores.
O limiar nativo ainda será selecionado no desenvolvimento; 0,7 é fixture.
Não há alertas duráveis nesta CLI. A integração e a reserva permanecem pendentes.

Conferir [política e critérios](../../docs/operations/CAUSAL_REPLAY.md#política-operacional-de-triagem)
e o [roadmap](../../docs/project/ROADMAP.md#próxima-entrega-concreta) antes de executar cada incremento.
Paridade, utilidade da fila e latência/custo terão evidências distintas.

O seletor do limiar está preparado em `streaming.preparation.development_alert_threshold`.
Executar somente depois de commitar o incremento; conferir [seleção e verificação](../../docs/operations/CAUSAL_REPLAY.md#selecionar-e-verificar-o-limiar-no-desenvolvimento).
Ele reutiliza as 21 partições autorizadas da Gold e a release congelada, sem alvos na seleção.
Ainda não recebemos um limiar nativo; integração durável e setembro permanecem pendentes.
